In [1]:
%run StdPackages.ipynb
d['processedData'] = os.path.join(d['data'],'processedData') # update to raw data folder
os.chdir(d['py'])
from loadIO import *
import RAS
from scipy.cluster.vq import kmeans2

No clean-up of work-folder


# Data for MEEOpt 

Aggregate into two industries, one durable. Further adjustments are made in specific implementations.

## 1. Aggregate data

*Load data:*

In [2]:
t0 = 2019
fullIO = os.path.join(d['processedData'], f'IO{t0}_s146_dur7')
name = f'IO_OPTdb' # add new name for the database when exporting
db = GpyDB(fullIO, name = name)

*Map to two random industries production industries and one durable:*

In [3]:
m = pd.MultiIndex.from_product([db('s_p')[:70], ['agg1']], names = ['s','ss']).union(
    pd.MultiIndex.from_product([db('s_p')[70:], ['agg2']], names = ['s','ss']))
mDur = pd.MultiIndex.from_tuples([(k, 'K') for k in db('s_i')], names = ['n','nn'])
mDur_s = mDur.set_levels(mDur.levels[-1].map(lambda x: 'I_'+x), level = 1).rename(['s','ss'])
# create full mapping of index 's':
ms = m.union(mDur_s)
ms_neutral = pd.MultiIndex.from_arrays([db('s'), db('s').rename('ss')]) # neutral mapping from (x,x)
m_sector = ms.union(adj.rc_pd(ms_neutral, ('not', ms.levels[0]))) # full mapping: Use specific mapping if this exists, otherwise default to neutral mapping.

*Apply mapping:*

In [4]:
AggDB.aggDB(db, m_sector);

*Adjust goods index:*

In [5]:
mn = m.union(m.set_levels([l+'_F' for l in m.levels])).rename(['n','nn']).union(mDur)
mn_neutral = pd.MultiIndex.from_arrays([db('n'), db('n').rename('nn')])
m_goods = mn.union(adj.rc_pd(mn_neutral, ('not', mn.levels[0])))

*Apply mapping:*

In [6]:
AggDB.aggDB(db, m_goods);

## 2. Model data

*Clean up some data:*

In [7]:
[db.series.database.pop(k) for k in ('gc','vC','vC_tax')];
[db.__setitem__(k, db(k)[db(k)!=0]) for k in db.getTypes(['var'])];

Scale to 100 billion DKK (instead of million DKK):

In [8]:
factor = 1e5
[db.__setitem__(k, db(k)/factor) for k in db.getTypes(['var']) if k not in ['qCO2', 'M1990','gCO2']];
db._scale = db._scale * factor # Note: The ._scale attribute indicates the value of 1 unit in the database.

Translate depreciation of durables to rates, distinguish between investments and durables (flow, stock) with investment good syntax ```I_x``` for durable ```x```. Define mapping ```dur2inv``` and subsets ```dur_p, inv_p```. Add investments and value of durables to the vector ```vD```:

In [9]:
db['rDepr'] = db('vD_depr')/db('vD_dur')
db['dur_p'] = db('vD_dur').index.levels[db['vD_dur'].domains.index('n')]
db['inv_p'] = db('dur_p').map(lambda x: f'I_{x}')
db['dur2inv'] = pd.MultiIndex.from_arrays([db('dur_p'), db('inv_p').rename('nn')])
db('vD_inv').index = db('vD_inv').index.set_levels(db('vD_inv').index.levels[db['vD_inv'].domains.index('n')].map(lambda x: f'I_{x}'), level = 'n')
db['vD'] = db('vD_inv').combine_first(db('vD')).combine_first(db('vD_dur'))

### RAS

Simple RAS algorithm:

In [10]:
threshold = 0 # negative values
v0 = adj.rc_pd(db('vD'), ('and', [('or', [db('n_p'), db('n_F')]),
                                  ('or', [db('s_p'), db('s_i')])]))
leaveCols = db('n_F') # are there any type of goods that we do not need to balance
leaveRows = None # are there any type of sectors that we do not need to balance
vBar = v0[v0<threshold] * 0

*Get RAS adjustments:*

In [11]:
vD = RAS.simpleRAS(v0, vBar, leaveCols = leaveCols, leaveRows = leaveRows, tol = 1e-8, iterMax = 1000)

Largest deviation summing over n: 0.0
Largest deviation summing over s: 0.0


*Merge things back up again:*

In [12]:
vD_full = vD.combine_first(db('vD'))
vD_full = vD_full[vD_full!=0] # remove zero values again

*Remove residual income category (we don't currently use this in the model, this will enter the return on durables instead):*

In [13]:
# db['vD'] = adj.rc_pd(vD_full, ('not', pd.Index(['resIncome'], name = 'n')))

### Create other variables

In [14]:
db['R_LR'] = gpy(1.05, name = 'R_LR', type = 'par')
db['infl_LR'] = gpy(0.0, name = 'infl_LR', type = 'par')
db['g_LR'] = gpy(0.02, name = 'g_LR', type = 'par')
model_vS(db)
model_p(db)
model_durables(db, db('R_LR'), db('infl_LR'))
model_quantNonDurables(db) 

### Create other subsets and mappings

Subsets of goods/sectors:

In [15]:
db['nEqui'] = db('vS').index.droplevel('s').unique() # what goods require an equilibrium condition
db['d_qS'] = db['vS'].index 
db['d_qD'] = adj.rc_pd(db('vD'), db('nEqui')).index 
db['d_qSEqui'] = adj.rc_pd(db['d_qS'].vals, ('not', db('s_HH'))) # Subset of qS values to be endogenized in general equilibrium
db['d_pEqui'] = pd.Index(['L'], name ='n') # Subset of prices to be endogenized in general equilibrium 

### Export

In [16]:
AggDB.updSetsFromSyms(db, types = ['var','par','map']) # define sets from variables/parameters defined throughout
db.export()